# MapReduce WordCount — Interactive Lab

So far we only **stored** data in HDFS. This lab finally does the other half of Hadoop's promise — **bringing computation to the data**. We run the canonical *WordCount* MapReduce job over a large text corpus and watch YARN schedule it across the cluster.

No Java to write: Hadoop ships a `hadoop-mapreduce-examples` jar with `wordcount` built in. We just point it at an HDFS input and let YARN do the rest.

**Dataset:** public-domain books from [Project Gutenberg](https://www.gutenberg.org) — no authentication. Scale the corpus by raising `BOOK_IDS`.

**What you'll watch:** the job appear in the [ResourceManager UI (localhost:8088)](http://localhost:8088) and move through *Map → Shuffle → Reduce*.

## Setup

In [ ]:
import subprocess
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to HDFS via proxy ✓')


def hadoop(cmd: str, container: str = 'namenode', timeout: int = 900) -> str:
    """Run a command inside a cluster container (for yarn/hdfs admin ops)."""
    try:
        p = subprocess.run(['docker', 'exec', container, 'bash', '-lc', cmd],
                           capture_output=True, text=True, timeout=timeout)
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        print(f'⚠️  Could not run docker exec ({e}).')
        print(f'    Run manually inside `make shell-{container}`:\n    {cmd}')
        return ''
    if p.returncode != 0:
        print(f'⚠️  Exit {p.returncode}. stderr tail:\n{p.stderr[-2000:]}')
    return p.stdout + p.stderr

## 1. Download the corpus

Each book is a plain‑text file. **Start with a few** to validate, then add more IDs to grow the corpus. War and Peace, Les Misérables and Monte Cristo are each ~2–3 MB on their own.

In [ ]:
import os
import requests

os.makedirs('../temp/books', exist_ok=True)

# Public-domain Project Gutenberg IDs. Add more to scale the corpus volume.
BOOK_IDS = [1342, 84, 1661, 2701, 1400, 98, 174, 345, 2600, 1260,
            135, 1184, 4300, 158, 11]
GUTENBERG = 'https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt'

local_books = []
for bid in BOOK_IDS:
    dest = f'../temp/books/pg{bid}.txt'
    if not os.path.exists(dest):
        try:
            r = requests.get(GUTENBERG.format(id=bid), timeout=60)
            r.raise_for_status()
            with open(dest, 'wb') as fh:
                fh.write(r.content)
        except requests.RequestException as e:
            print(f'  skip {bid}: {e}')
            continue
    local_books.append(dest)

total_mb = sum(os.path.getsize(f) for f in local_books) / 1e6
print(f'{len(local_books)} books, {total_mb:.1f} MB total')

## 2. Ingest the corpus into HDFS

MapReduce reads its input **from HDFS**, where it is already split into blocks across DataNodes — each block becomes a candidate input split processed by a map task running near its data.

In [ ]:
INPUT_DIR = '/datasets/text'
client.makedirs(INPUT_DIR, permission=0o755)

for book in local_books:
    client.upload(f"{INPUT_DIR}/{os.path.basename(book)}", book, overwrite=True)

files = client.list(INPUT_DIR)
print(f'{len(files)} files in {INPUT_DIR}')

## 3. Locate the examples jar

The jar ships inside the Hadoop image; its exact filename includes the version, so we resolve it with a glob instead of hard-coding it.

In [ ]:
EXAMPLES_JAR = hadoop(
    'ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar'
).strip().splitlines()[-1]
print('Examples jar:', EXAMPLES_JAR)

## 4. Run the WordCount job on YARN

### CLI equivalent
```bash
yarn jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar \
    wordcount /datasets/text /output/wordcount
```

The output directory **must not already exist** — we remove it first. Open the [ResourceManager UI (localhost:8088)](http://localhost:8088) **now**, then run the cell: the application shows up live with its map/reduce progress.

In [ ]:
OUTPUT_DIR = '/output/wordcount'

# Remove any previous output (idempotent re-runs)
try:
    client.delete(OUTPUT_DIR, recursive=True)
except Exception:
    pass

log = hadoop(f'yarn jar {EXAMPLES_JAR} wordcount {INPUT_DIR} {OUTPUT_DIR}')
# Show the tail: counters (map input records, reduce output records, etc.)
print(log[-2500:])

The counters at the end (`Map input records`, `Reduce output records`, `Bytes Read`) are the receipt that the job actually fanned out over the cluster. In the [UI](http://localhost:8088) the application state should read **`FINISHED` / `SUCCEEDED`**.

## 5. Read the results back into Pandas

WordCount writes tab-separated `word<TAB>count` lines into `part-r-*` files. We read them through the proxy and rank the most frequent words.

In [ ]:
import io
import pandas as pd

# There may be several part files; concatenate them all.
parts = [f for f in client.list(OUTPUT_DIR) if f.startswith('part-')]
frames = []
for part in parts:
    with client.read(f'{OUTPUT_DIR}/{part}') as reader:
        frames.append(pd.read_csv(io.BytesIO(reader.read()), sep='\t',
                                  names=['word', 'count'], quoting=3))

wc = pd.concat(frames, ignore_index=True)
print(f'Distinct words: {len(wc):,}')

# Drop very short tokens to surface meaningful words
top = (wc[wc['word'].str.len() > 4]
         .sort_values('count', ascending=False)
         .head(20).reset_index(drop=True))
top

## 6. Cleanup

In [ ]:
import shutil

client.delete(INPUT_DIR, recursive=True)
client.delete(OUTPUT_DIR, recursive=True)
print('Removed HDFS input/output ✓')

shutil.rmtree('../temp/books', ignore_errors=True)
print('Removed local books ✓')

## Summary

| Step | What it demonstrated |
|---|---|
| Ingest corpus to HDFS | input is pre-split into blocks across DataNodes |
| `yarn jar ... wordcount` | YARN schedules map/reduce tasks **on the cluster** |
| ResourceManager UI (8088) | live Map → Shuffle → Reduce, `SUCCEEDED` state |
| Read `part-r-*` back | results land in HDFS, consumed with Pandas |

Next: **`06_hadoop_streaming_weather.ipynb`** runs MapReduce with **your own Python** mapper/reducer instead of a prebuilt jar.